In [1]:
import numpy as np, zarr
import dask.array as da

SCALE = (1.0, 4.0, 1.0, 1.0)

VIDEOS = [
    # "44b6_33b596bf", "44b6_40c45f5a", "44b6_551a5dba", # used in baseline (v0)
    "6bba_61dd1e0d", # added in first iter (v1)
]

In [2]:
def get_geff(path):
    g = zarr.open(path, mode='r')
    
    points = np.column_stack([
        g["nodes/props/t/values"][:],
        g["nodes/props/z/values"][:],
        g["nodes/props/y/values"][:],
        g["nodes/props/x/values"][:],
    ])
    return points

In [5]:
import napari, numpy as np, zarr, os


for vid in VIDEOS:
    video = da.from_zarr(f'unet_patches/{vid}.zarr/0')
    # proc = da.from_zarr(f'../test_processed/{vid}.zarr/0')
    pts = np.load(f'unet_labels/{vid}_pts.npy')
    watersheds = np.load(f"heatmaps/{vid}_v3_watersheds.npy")

    viewer = napari.Viewer(ndisplay=3, title=vid)
    try:
        viewer.axes.visible = viewer.axes.colored = viewer.axes.labels = True
    except AttributeError:
        a = viewer.scene.overlays.axes
        a.visible = a.colored = a.labels = True
    
    viewer.add_image(
        video, 
        name='orig_vol',
        scale=SCALE, 
        colormap='magma', 
        blending='additive'
    )

    viewer.add_image(
        watersheds, 
        name='watershed',
        scale=SCALE, 
        colormap='magma', 
        blending='additive'
    )
    
    viewer.add_points(
        pts,
        name='gt',
        scale=SCALE,
        face_color='cyan',
        size=4,
    )
    napari.run()


could not determine DPI
